# Picoclimatic Synthetic Data: Shapelet Clustering + Explanation Stability

This notebook:
1. Generates synthetic picoclimatic data (T=4 timesteps, V=23 variables per station-day)
2. Tests mixed-length vs fixed-length shapelets
3. Quantifies explanation stability across 7 seeds
4. Demonstrates Direction C: bounded sampling for speed

In [ ]:
import os, sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

os.environ.setdefault('OMP_NUM_THREADS', '1')

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

repo_root = Path.cwd()
if not (repo_root / 'scripts').exists() and (repo_root.parent / 'scripts').exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / 'scripts' / 'data'))
print(f'Repo root: {repo_root}')

In [ ]:
from generate_picoclimate_data import generate_dataset

data_dir = repo_root / 'data' / 'picoclimate_test'
data_dir.mkdir(parents=True, exist_ok=True)

raw_path, win_path, meta_path, fields_path = generate_dataset(
    outdir=data_dir,
    seed=42,
    city='Nantes',
    n_locations=12,
    days=30,
)

print(f'Generated synthetic picoclimate data')
print(f'  Window features: {win_path}')

In [ ]:
win_df = pd.read_csv(win_path)

slot_cols = [c for c in win_df.columns if '__slot_' in c]
print(f'Found {len(slot_cols)} slot columns (4 slots x 23 variables)')

X_flat = win_df[slot_cols].to_numpy(dtype=float)
print(f'Data shape: {X_flat.shape}')

if 'true_regime' in win_df.columns:
    y_true = win_df['true_regime'].to_numpy()

scaler = StandardScaler()
X = scaler.fit_transform(X_flat)
print(f'Standardized X: {X.shape}')